Embedding using DistilBert and training using CNN

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
data = pd.read_csv("training_data_lowercase.csv",sep='\t', names=['label', 'title'])
print(data.shape)
data.fillna("",inplace=True)
print(data.head())


(34152, 2)
   label                                              title
0      0  donald trump sends out embarrassing new year‚s...
1      0  drunk bragging trump staffer started russian c...
2      0  sheriff david clarke becomes an internet joke ...
3      0  trump is so obsessed he even has obama‚s name ...
4      0  pope francis just called out donald trump duri...


as part of pre proc we are able to see color codes in csv which are incorrectly interpretted by vs code
they are not color code but simply corresponds to episode num

we also see video/picture are there in some data points, while these are just metadata it could change the meaning of the sentence once we remove punctuations

we also see that this metadata in enclosed in () in Training sample while its enclosed in [] in testing data, so we might need different pre processing

In [5]:
from preProc import normalize_text

data["clean_text"] = data["title"].apply(normalize_text)
print(data.head)

<bound method NDFrame.head of        label                                              title  \
0          0  donald trump sends out embarrassing new year‚s...   
1          0  drunk bragging trump staffer started russian c...   
2          0  sheriff david clarke becomes an internet joke ...   
3          0  trump is so obsessed he even has obama‚s name ...   
4          0  pope francis just called out donald trump duri...   
...      ...                                                ...   
34147      1  tears in rain as thais gather for late king's ...   
34148      1  pyongyang university needs non-u.s. teachers a...   
34149      1  philippine president duterte to visit japan ah...   
34150      1  japan's abe may have won election\tbut many do...   
34151      1  demoralized and divided: inside catalonia's po...   

                                              clean_text  
0      donald trump sends out embarrassing new year s...  
1      drunk bragging trump staffer started rus

In [6]:
from sklearn.model_selection import train_test_split
X=data.drop(columns=['label'])
y=data['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_val.shape}")
print(f"Training output size: {y_train.shape}")
print(f"Testing output size: {y_val.shape}")

Training set size: (27321, 2)
Testing set size: (6831, 2)
Training output size: (27321,)
Testing output size: (6831,)


DistilBERT Embeddings

In [9]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch
import numpy as np

tokenizer  = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')

distilbert.eval()

def get_distilbert_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')

        with torch.no_grad():
            outputs = distilbert(**inputs)

        embeddings = outputs.last_hidden_state.mean(dim=1).numpy()
        all_embeddings.append(embeddings)

        if i % 500 == 0:
            print(f"Encoded {i}/{len(texts)}")

    return np.vstack(all_embeddings)

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

X_train_EMB = get_distilbert_embeddings(X_train["clean_text"].tolist())
X_val_EMB   = get_distilbert_embeddings(X_val["clean_text"].tolist())
X_train_CNN = X_train_EMB.reshape(X_train_EMB.shape[0], X_train_EMB.shape[1], 1)
X_val_CNN = X_val_EMB.reshape(X_val_EMB.shape[0], X_val_EMB.shape[1], 1)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoded 0/27321
Encoded 4000/27321
Encoded 8000/27321
Encoded 12000/27321
Encoded 16000/27321
Encoded 20000/27321
Encoded 24000/27321
Encoded 0/6831
Encoded 4000/6831


In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


model_bin_sin = Sequential([
     layers.Input(shape=(768,1)),
     layers.Conv1D(filters=128, kernel_size=3, activation='relu'),
     layers.MaxPooling1D(pool_size=2),
     layers.Dropout(0.3),
     layers.Bidirectional(layers.LSTM(64, return_sequences=False)),
     layers.Dropout(0.3),
     layers.Dense(64, activation='relu'),
     layers.Dense(1, activation='sigmoid')

])

model_bin_sin.summary()

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

model_bin_sin.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

trainer = model_bin_sin.fit(
    X_train_CNN, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_CNN, y_val),
    verbose=1,
    callbacks=callbacks
)

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_9 (Conv1D)               │ (None, 766, 128)       │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 383, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 383, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 107,649 (420.50 KB)

 Trainable params: 107,649 (420.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
854/854 ━━━━━━━━━━━━━━━━━━━━ 31s 30ms/step - accuracy: 0.7960 - loss: 0.4403 - val_accuracy: 0.8485 - val_loss: 0.3533 - learning_rate: 0.0010
Epoch 2/20
854/854 ━━━━━━━━━━━━━━━━━━━━ 25s 30ms/step - accuracy: 0.8480 - loss: 0.3513 - val_accuracy: 0.8656 - val_loss: 0.3164 - learning_rate: 0.0010
Epoch 3/20
854/854 ━━━━━━━━━━━━━━━━━━━━ 25s 30ms/step - accuracy: 0.8587 - loss: 0.3222 - val_accuracy: 0.8795 - val_loss: 0.2854 - learning_rate: 0.0010
Epoch 4/20
854/854 ━━━━━━━━━━━━━━━━━━━━ 25s 30ms/step - accuracy: 0.8697 - loss: 0.3049 - val_accuracy: 0.8773 - val_loss: 0.2915 - learning_rate: 0.0010
Epoch 5/20
854/854 ━━━━━━━━━━━━━━━━━━━━ 25s 30ms/step - accuracy: 0.8781 - loss: 0.2908 - val_accuracy: 0.8883 - val_loss: 0.2668 - learning_rate: 0.0010
Epoch 6/20
854/854 ━━━━━━━━━━━━━━━━━━━━ 26s 30ms/step - accuracy: 0.8838 - loss: 0.2739 - val_accuracy: 0.9005 - val_loss: 0.2535 - learning_rate: 0.0010
Epoch 7/20
854/854 ━━━━━━━━━━━━━━━━━━━━ 26s 30ms/step - accuracy: 0.8892 - l